### Importing of libraries

In [4]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression 
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, root_mean_squared_error
import sqlite3
%matplotlib inline
import math
import xgboost as xgb
from sklearn.preprocessing import MinMaxScaler
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.regularizers import l1_l2
import warnings
warnings.filterwarnings('ignore')

### Data Import

In [5]:
db_path = r"E:\Machine Learning Research\DataModelling\IDA_Data.db"  # <-- change this to your actual .db file path
table_name = "IDA_Data"                    # <-- your table name

conn = sqlite3.connect(db_path)
df = pd.read_sql(f"SELECT * FROM {table_name}", conn)
conn.close()

# === Display===
# print(df.shape)
# print(list(df.columns))
# === Optionally, show first few rows ===
# print(InputHeads)
# print(OutputHeads)
# print("\nFirst 5 rows:")
# print(df.columns)
# df.columns

AllColumns = ['id', 'Earthquake', 'ScaleFactor', 'Building', 'BaseCondition', 'ly-Ly', 'lx-Lx', 'ly-Ly-lx-Lx', 'ly-lx-', 'lx-ly-', 'lx-ly--ly-lx-', 'Plan-area', 'Seismic-weight', 
              'StiffnessX_Story5', 'StiffnessX_Story4', 'StiffnessX_Story3', 'StiffnessX_Story2', 'StiffnessX_Story1', 'StiffnessX_Total', 'Layer1_FrictionA', 'Layer1_G_kPa', 
              'Layer1_E_kPa', 'Layer1_B_kPa', 'Layer1_SpGr', 'Layer1_Cohesion', 'Layer2_FrictionA', 'Layer2_G_kPa', 'Layer2_E_kPa', 'Layer2_B_kPa', 'Layer2_SpGr', 'Layer2_Cohesion',
              'Layer3_FrictionA', 'Layer3_G_kPa', 'Layer3_E_kPa', 'Layer3_B_kPa', 'Layer3_SpGr', 'Layer3_Cohesion', 'PGA', 'Magnitude', 'Mechanism', 'Rjb', 'Rrup', 'Vs30', 'cav_gs', 
              'scav_gs', 'bcav_gs', 'arias_mps', 'husid_s', 'spi_mps', 'hous_m', 'maxacceleration_mps2', 'maxvelocity_mps', 'maxdisplacement_m', 'maxpsd_cmps', 'maxsa_mps2', 'maxpsv_mps', 
              'Fundamental_Period', 'Drift-X_Level-1', 'Drift-X_Level-2', 'Drift-X_Level-3', 'Drift-X_Level-4', 'Drift-X_Level-5', 'Drift-X_Level-6', 'Drift-Y_Level-1', 'Drift-Y_Level-2',
              'Drift-Y_Level-3', 'Drift-Y_Level-4', 'Drift-Y_Level-5', 'Drift-Y_Level-6', 'Displacement-X_Level-1', 'Displacement-X_Level-2', 'Displacement-X_Level-3', 'Displacement-X_Level-4',
              'Displacement-X_Level-5', 'Displacement-X_Level-6', 'Displacement-Y_Level-1', 'Displacement-Y_Level-2', 'Displacement-Y_Level-3', 'Displacement-Y_Level-4', 'Displacement-Y_Level-5',
              'Displacement-Y_Level-6', 'Reaction-Force-X_Level-1', 'Reaction-Force-X_Level-2', 'Reaction-Force-X_Level-3', 'Reaction-Force-X_Level-4', 'Reaction-Force-X_Level-5', 'Reaction-Force-X_Level-6', 
              'Reaction-Force-Y_Level-1', 'Reaction-Force-Y_Level-2', 'Reaction-Force-Y_Level-3', 'Reaction-Force-Y_Level-4', 'Reaction-Force-Y_Level-5', 'Reaction-Force-Y_Level-6', 'Reaction-Moment-X_Level-1',
              'Reaction-Moment-X_Level-2', 'Reaction-Moment-X_Level-3', 'Reaction-Moment-X_Level-4', 'Reaction-Moment-X_Level-5', 'Reaction-Moment-X_Level-6', 'Reaction-Moment-Y_Level-1',
              'Reaction-Moment-Y_Level-2', 'Reaction-Moment-Y_Level-3', 'Reaction-Moment-Y_Level-4', 'Reaction-Moment-Y_Level-5', 'Reaction-Moment-Y_Level-6', 'Rotation-X_Level-1', 'Rotation-X_Level-2',
              'Rotation-X_Level-3', 'Rotation-X_Level-4', 'Rotation-X_Level-5', 'Rotation-X_Level-6', 'Rotation-Y_Level-1', 'Rotation-Y_Level-2', 'Rotation-Y_Level-3', 'Rotation-Y_Level-4', 'Rotation-Y_Level-5',
              'Rotation-Y_Level-6', 'Rotation-Z_Level-1', 'Rotation-Z_Level-2', 'Rotation-Z_Level-3', 'Rotation-Z_Level-4', 'Rotation-Z_Level-5', 'Rotation-Z_Level-6', 'Torsional-Irregularity-Ratio_Level-1',
              'Torsional-Irregularity-Ratio_Level-2', 'Torsional-Irregularity-Ratio_Level-3', 'Torsional-Irregularity-Ratio_Level-4', 'Torsional-Irregularity-Ratio_Level-5', 'Torsional-Irregularity-Ratio_Level-6',
              'Max-Uplift_Level-1', 'Max-Uplift-Point_Level-1', 'Max-Settlement_Level-1', 'Max-Settlement-Point_Level-1', 'Max-Pseudo-Time_Level-1']

### Data Processing

In [6]:
def dataPreparation(DoNormalize = True,Z_ScoreScaler = True    ,        ModelSet = 2     ,includeCategoricalData = False  ):
    
    # Creating of the PeakGroundAcceleration column on the database by product of ScaleFactor and MaximumAcceleration of the data
    df['maxacceleration_mps2'] = pd.to_numeric(df['maxacceleration_mps2'], errors='coerce')
    df['ScaleFactor'] = pd.to_numeric(df['ScaleFactor'], errors='coerce')
    df['PeakGroundAcceleration'] = df['maxacceleration_mps2'] * df['ScaleFactor']
    
    #Creating of the Prediction column as Max drift of the all levels merger in single column as below
    drift_cols = [    'Drift-X_Level-1', 'Drift-X_Level-2', 'Drift-X_Level-3',    'Drift-X_Level-4', 'Drift-X_Level-5', 'Drift-X_Level-6']
    df[drift_cols] = df[drift_cols].apply(pd.to_numeric, errors='coerce')
    df['Max_Drift_X'] = df[drift_cols].max(axis=1)
    
    # Check for any NaNs that might have appeared due to conversion issues
    # print(df[['maxacceleration_mps2', 'ScaleFactor', 'PeakGroundAcceleration']].sample(50))
    # print("Number of NaNs in new column:", df['PeakGroundAcceleration'].isna().sum())
    
    InputHeadsAvailableAll = ['Earthquake', 'ScaleFactor', 'Building', 'BaseCondition', 'ly-Ly', 'lx-Lx', 'ly-Ly-lx-Lx', 'ly-lx-', 'lx-ly-', 'lx-ly--ly-lx-', 'Plan-area', 'Seismic-weight', 
                  'StiffnessX_Story5', 'StiffnessX_Story4', 'StiffnessX_Story3', 'StiffnessX_Story2', 'StiffnessX_Story1', 'StiffnessX_Total', 
                  'Layer1_FrictionA', 'Layer1_G_kPa', 'Layer1_E_kPa', 'Layer1_B_kPa', 'Layer1_SpGr', 'Layer1_Cohesion',
                  'Layer2_FrictionA', 'Layer2_G_kPa', 'Layer2_E_kPa', 'Layer2_B_kPa', 'Layer2_SpGr', 'Layer2_Cohesion', 
                  'Layer3_FrictionA', 'Layer3_G_kPa', 'Layer3_E_kPa', 'Layer3_B_kPa', 'Layer3_SpGr', 'Layer3_Cohesion', 
                  'PGA', 'Magnitude', 'Mechanism', 'Rjb', 'Rrup', 'Vs30', 'cav_gs', 'scav_gs', 'bcav_gs', 'arias_mps', 'husid_s', 'spi_mps', 'hous_m', 'maxacceleration_mps2',
                  'maxvelocity_mps', 'maxdisplacement_m', 'maxpsd_cmps', 'maxsa_mps2', 'maxpsv_mps', 
                  'Fundamental_Period']
    InputHeadsSelected = ['ly-Ly', 'lx-Lx', 'ly-Ly-lx-Lx', 'ly-lx-', 'lx-ly-', 'lx-ly--ly-lx-',
                   'StiffnessX_Total', 'Plan-area', 'Seismic-weight', 
                  'Layer1_FrictionA', 'Layer1_G_kPa', 'Layer1_E_kPa', 'Layer1_B_kPa', 'Layer1_SpGr', 'Layer1_Cohesion',
                  'Layer2_FrictionA', 'Layer2_G_kPa', 'Layer2_E_kPa', 'Layer2_B_kPa', 'Layer2_SpGr', 'Layer2_Cohesion', 
                  'Layer3_FrictionA', 'Layer3_G_kPa', 'Layer3_E_kPa', 'Layer3_B_kPa', 'Layer3_SpGr', 'Layer3_Cohesion',                           
                  "PeakGroundAcceleration", 'Magnitude', 'arias_mps','maxpsd_cmps', 'maxsa_mps2', 'maxpsv_mps',  'cav_gs', 
                             'Fundamental_Period']
    # InputHeadsSelected = InputHeadsAvailableAll

    
    #Best performer for the fixedbase and displacement output
                   # "PeakGroundAcceleration", 'Magnitude', 'arias_mps','maxpsd_cmps', 'maxsa_mps2', 'maxpsv_mps',  'cav_gs', 
                   #           'Fundamental_Period']
    #Best performer for the maximum drift along X with r2 as 89.36 (Donot include  'Plan-area', 'Seismic-weight', 'StiffnessX_Total', as they donot produce any effect and only add burden to model)
    # InputHeadsSelected = ['ly-Ly', 'lx-Lx', 'ly-Ly-lx-Lx', 'ly-lx-', 'lx-ly-', 'lx-ly--ly-lx-',
                   
    #               'Layer1_FrictionA', 'Layer1_G_kPa', 'Layer1_E_kPa', 'Layer1_B_kPa', 'Layer1_SpGr', 'Layer1_Cohesion',
    #               'Layer2_FrictionA', 'Layer2_G_kPa', 'Layer2_E_kPa', 'Layer2_B_kPa', 'Layer2_SpGr', 'Layer2_Cohesion', 
    #               'Layer3_FrictionA', 'Layer3_G_kPa', 'Layer3_E_kPa', 'Layer3_B_kPa', 'Layer3_SpGr', 'Layer3_Cohesion', 
                          
    #               "PeakGroundAcceleration", 'Magnitude', 'arias_mps','maxpsd_cmps', 'maxsa_mps2', 'maxpsv_mps',  'cav_gs', 
    #                          'Fundamental_Period']
    #Best performer for the fixed and output as Reaction-Force-X_Level-1:  r2 as 0.95 max with 15 features
    # InputHeadsSelected = ['Earthquake', 'Building', 'Fundamental_Period',  'ly-Ly', 'lx-Lx', 'ly-Ly-lx-Lx',
    #               'Layer1_FrictionA', 'Layer1_G_kPa', 'Layer1_E_kPa', 'Layer1_B_kPa', 'Layer1_SpGr', 'Layer1_Cohesion',
    #               'Layer2_FrictionA', 'Layer2_G_kPa', 'Layer2_E_kPa', 'Layer2_B_kPa', 'Layer2_SpGr', 'Layer2_Cohesion', 
    #               'Layer3_FrictionA', 'Layer3_G_kPa', 'Layer3_E_kPa', 'Layer3_B_kPa', 'Layer3_SpGr', 'Layer3_Cohesion', 
    #                "PeakGroundAcceleration", 'arias_mps','maxpsd_cmps', 'maxsa_mps2', 'maxpsv_mps',
    #                         ]
    #For soil analysis take Rup, Vs30... Use maxacceleration_mps2 that represents the pga value of site and for the building specific analysis ie performance
    # Levels measurement take PSA, PSV and PSD values over the peac acc peak vel and peak disp and so on 
    #Final selected features (9): ['ScaleFactor', 'Vs30', 'cav_gs', 'arias_mps', 'husid_s', 'hous_m', 'maxvelocity_mps', 'maxdisplacement_m', 'maxpsd_cmps']
    
    
    
    #_______________________________________________________________________________________________________________________________________________________________Processing starts from here
    if ModelSet == 1 :
        InputHeads = [x for x in InputHeadsSelected if not x.startswith("Layer")] 
    elif ModelSet == 3:
        InputHeads = [x for x in InputHeadsSelected if not x.startswith("Layer")]
        InputHeads = InputHeads + ["BaseCondition"]
    else:
        InputHeads = InputHeadsSelected
    # OutputHeads = ['Displacement-X_Level-5']
    # OutputHeads = ["Reaction-Force-X_Level-1"]
    OutputHeads = ['Max_Drift_X']
    
    # df[InputHeads].info
    
    ####__________________________________________________________________________________________________________________________________Data selection 
    
    # Extract all the data from the fixed base conditions only
    if ModelSet == 1:
        df_fixed = df[df['BaseCondition'] == 'Fixed']
    elif ModelSet == 2:
        df_fixed = df[df['BaseCondition'] != 'Fixed']
    else:
        df_fixed = df[df['BaseCondition'].isin(['Fixed', 'Soft', 'Medium', 'Hard'])]
        
    
    #Convert all the data in database to numerical format
    if includeCategoricalData:
        df_numeric = df_fixed.copy()
        for col in df_numeric.columns:
            df_numeric[col] = pd.to_numeric(df_numeric[col], errors='ignore')
    else:
        df_numeric = df_fixed.apply(pd.to_numeric, errors='coerce')
        df_numeric.shape
    # df_numeric.head
    
    ##Extract the data if the scale factor is less than or equal to 1 only
    if ModelSet != 1:
        df_numeric = df_numeric[df_numeric['ScaleFactor'] <= 1]
    df_numeric = df_numeric[df_numeric['ScaleFactor'] <= 1]
    
    #Removing all the white spaces on the column titles
    df_numeric.columns = df_numeric.columns.str.replace(" ", "_")

    ####__________________________________________________________________________________________________________________________________Encoding of data
    # # Normalize for the data of the features as seismic weights, and so on
    features_to_normalize = [x for x in InputHeads
                             if pd.api.types.is_numeric_dtype(df_numeric[x])]
    
    
    #One hot encoding of the categorical data if user set it to include in the model
    if includeCategoricalData:
        Catcolumns =  [x for x in InputHeads if x not in features_to_normalize]
        # print(Catcolumns)

        if Catcolumns:
            data_toEncode = df_numeric[Catcolumns]     
            CatDataEncoded = pd.get_dummies(data_toEncode, columns = Catcolumns,  dtype=float)
            df_numeric = df_numeric.drop(columns=Catcolumns)

        
            if df_numeric.shape[0] == CatDataEncoded.shape[0]:
                InputHeads = InputHeads + list(CatDataEncoded.columns)
                df_numeric = pd.concat([df_numeric, CatDataEncoded], axis=1)

            else: 
                raise
    

    
    ## Converts all the data to the numeric even to that of hot encoded as they are in float
    df_numeric = df_numeric.apply(pd.to_numeric, errors='coerce')
        
    
    ####__________________________________________________________________________________________________________________________________Missing Data Handling
    df_numeric =df_numeric.dropna(axis=1, how='all')             #Dropping for all the columns (axis = 1) if it has all values of NaN
    
    #### Show only columns that actually contain NaN
    # nan_counts = df_numeric.isna().sum()
    # nan_counts = nan_counts[nan_counts > 0]
    # print("Columns containing NaN values (column: count):")
    # print(nan_counts)
    
    # nan_rows = df_numeric[df_numeric.isna().any(axis=1)]
    # print("Rows containing NaN values:")
    # print(nan_rows)
    
    df_numeric = df_numeric.apply(lambda col: col.fillna(col.median()) if col.dtype != 'object' else col)   #For any value other than object if the data is shown as missing then it fills up with the median of that column

    # nan_counts = df_numeric.isna().sum()
    # nan_counts = nan_counts[nan_counts > 0]
    # print("Columns containing NaN values (column: count):")
    # print(nan_counts)
    
    df_numeric = df_numeric.dropna()                             #Now also if any of the Nan data persists then the system drops that specific rows
    

    ####__________________________________________________________________________________________________________________________________Normalization of data
    if DoNormalize:
        scaler = MinMaxScaler(feature_range=(0, 1))
        if Z_ScoreScaler:
            scaler = StandardScaler()
        cols_to_scale = [c for c in features_to_normalize if c in df_numeric.columns] # Only normalize columns that actually exist in df
        df_numeric[cols_to_scale] = scaler.fit_transform(df_numeric[cols_to_scale])

    ####__________________________________________________________________________________________________________________________________Assignment of the training data
    
    # # Extract for the data to be in x and y variables
    X = df_numeric[[col for col in InputHeads if col in df_numeric.columns]]
    y = df_numeric[[col for col in OutputHeads if col in df_numeric.columns]]

    df_numeric = pd.concat([X, y], axis=1)  # combine X and y

    
    # X.shape
    # print((X.columns))
    # cols = CatDataEncoded.columns
    # for co in cols:
    #     print(co)
    # print(CatDataEncoded.columns)
    # df_numeric.sample(10)
    # X.shape

    return X, y



### Getting data Statistics

In [7]:
import pandas as pd
import numpy as np

def generate_complete_data_statistics(X, y):
    """Generate comprehensive statistical summary for entire dataset"""
    
    # Convert to proper formats
    X_clean = X.apply(pd.to_numeric, errors='coerce')
    y_clean = y.values.ravel() if hasattr(y, 'values') else y
    
    print("COMPLETE DATASET STATISTICAL SUMMARY")
    print("="*80)
    print(f"{'Dataset':<20} {'Samples':<10} {'Features':<10}")
    print(f"{'':<20} {X_clean.shape[0]:<10} {X_clean.shape[1]:<10}")
    print("="*80)
    
    # Features summary
    print(f"\n{'FEATURES SUMMARY':<50}")
    print("-"*50)
    print(f"{'Parameter':<25} {'Count':<8} {'Mean':<12} {'Std':<12} {'Min':<10} {'Max':<10}")
    print("-"*50)
    
    for column in X_clean.columns:
        count = len(X_clean[column])
        mean = X_clean[column].mean()
        std = X_clean[column].std()
        min_val = X_clean[column].min()
        max_val = X_clean[column].max()
        
        print(f"{column:<25} {count:<8} {mean:<12.4f} {std:<12.4f} {min_val:<10.2f} {max_val:<10.2f}")

    # Target variable summary
    print(f"\n{'TARGET VARIABLE SUMMARY':<50}")
    print("-"*50)
    print(f"{'Parameter':<25} {'Count':<8} {'Mean':<12} {'Std':<12} {'Min':<10} {'Max':<10}")
    print("-"*50)
    
    count = len(y_clean)
    mean = y_clean.mean()
    std = y_clean.std()
    min_val = y_clean.min()
    max_val = y_clean.max()
    
    print(f"{'Target':<25} {count:<8} {mean:<12.4f} {std:<12.4f} {min_val:<10.2f} {max_val:<10.2f}")

    # Additional statistics
    print(f"\n{'ADDITIONAL STATISTICS':<50}")
    print("-"*50)
    print(f"{'Metric':<25} {'Value':<25}")
    print("-"*50)
    print(f"{'Total Samples':<25} {X_clean.shape[0]:<25}")
    print(f"{'Total Features':<25} {X_clean.shape[1]:<25}")
    print(f"{'Missing Values in X':<25} {X_clean.isnull().sum().sum():<25}")
    print(f"{'Missing Values in y':<25} {np.isnan(y_clean).sum():<25}")
    print(f"{'Target Skewness':<25} {pd.Series(y_clean).skew():<25.4f}")
    print(f"{'Target Kurtosis':<25} {pd.Series(y_clean).kurtosis():<25.4f}")

# Quick version for compact output
def quick_data_stats(X, y):
    """Quick statistical summary"""
    X_clean = X.apply(pd.to_numeric, errors='coerce')
    y_clean = y.values.ravel() if hasattr(y, 'values') else y
    
    print("DATASET STATISTICS")
    print("="*90)
    print(f"{'Parameter':<20} {'Count':<8} {'Mean':<12} {'Std':<12} {'Min':<10} {'Max':<10} {'Type':<10}")
    print("-"*90)
    
    # Features
    for column in X_clean.columns:
        data = X_clean[column]
        print(f"{column:<20} {len(data):<8} {data.mean():<12.4f} {data.std():<12.4f} "
              f"{data.min():<10.2f} {data.max():<10.2f} {'Feature':<10}")
    
    # Target
    print(f"{'Target':<20} {len(y_clean):<8} {y_clean.mean():<12.4f} {y_clean.std():<12.4f} "
          f"{y_clean.min():<10.2f} {y_clean.max():<10.2f} {'Target':<10}")

# Generate LaTeX table for research paper
def generate_latex_table(X, y):
    """Generate LaTeX table for research paper"""
    X_clean = X.apply(pd.to_numeric, errors='coerce')
    y_clean = y.values.ravel() if hasattr(y, 'values') else y
    
    print("\nLATEX TABLE FOR RESEARCH PAPER:")
    print("\\begin{table}[htbp]")
    print("\\centering")
    print("\\caption{Descriptive Statistics of the Complete Dataset}")
    print("\\label{tab:data_statistics}")
    print("\\begin{tabular}{lrrrrr}")
    print("\\hline")
    print("Parameter & Count & Mean & Std & Min & Max \\\\")
    print("\\hline")
    
    # Features
    for column in X_clean.columns:
        data = X_clean[column]
        print(f"{column} & {len(data)} & {data.mean():.4f} & {data.std():.4f} & {data.min():.2f} & {data.max():.2f} \\\\")
    
    # Target
    print(f"Target & {len(y_clean)} & {y_clean.mean():.4f} & {y_clean.std():.4f} & {y_clean.min():.2f} & {y_clean.max():.2f} \\\\")
    print("\\hline")
    print("\\end{tabular}")
    print("\\end{table}")



# Stacking of Regressors (xGBOOST, Catboost, NN Model)

In [6]:
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, KBinsDiscretizer

class EnhancedNeuralNetworkAnalyzer:
    def __init__(self, X, y, test_size=0.2, random_state=42, 
                 augment_data=False, noise_factor=0.01, stratify_data=False, n_bins=5):
        """Initialize enhanced neural network analyzer with augmentation and stratification"""
        self.X = X.copy()
        self.y = y.copy()
        self.test_size = test_size
        self.random_state = random_state
        self.augment_data = augment_data
        self.noise_factor = noise_factor
        self.stratify_data = stratify_data
        self.n_bins = n_bins
        self.scaler = StandardScaler()
        self.history = None
        self.model = None
        
        # Prepare data with enhanced features
        self._prepare_enhanced_data()
        
        print(f"Enhanced Neural Network Analyzer initialized:")
        print(f"Training data: {self.X_train_scaled.shape}")
        print(f"Testing data: {self.X_test_scaled.shape}")
        print(f"Data augmentation: {self.augment_data}")
        print(f"Stratified splitting: {self.stratify_data}")

    
    
    def _create_stratification_bins(self, y, n_bins=5):
        """Create bins for continuous target variable for stratification"""
        # Use KBinsDiscretizer to create balanced bins
        discretizer = KBinsDiscretizer(n_bins=n_bins, encode='ordinal', strategy='quantile')
        y_binned = discretizer.fit_transform(y.reshape(-1, 1)).ravel()
        return y_binned.astype(int)
    
    def add_data_augmentation(self, X_train, y_train, noise_factor=0.01):
        """Add Gaussian noise to training data for regularization"""
        print(f"🔧 Applying data augmentation with noise factor: {noise_factor}")
        
        # Add Gaussian noise
        X_noise = X_train + noise_factor * np.random.normal(
            loc=0.0, scale=1.0, size=X_train.shape
        )
        
        # Combine original and augmented data
        X_augmented = np.vstack([X_train, X_noise])
        y_augmented = np.concatenate([y_train, y_train])
        
        print(f"📈 Data augmented: {X_train.shape} -> {X_augmented.shape}")
        
        return X_augmented, y_augmented
    
    def _prepare_enhanced_data(self):
        """Enhanced data preparation with stratification and augmentation"""
        try:
            # Convert to numpy arrays
            if hasattr(self.X, 'values'):
                X_values = self.X.values
            else:
                X_values = np.array(self.X)
                
            if hasattr(self.y, 'values'):
                y_values = self.y.values
            else:
                y_values = np.array(self.y)
            
            y_values = y_values.ravel()
            
            # Stratified splitting for continuous target
            if self.stratify_data:
                print("🎯 Using stratified data splitting...")
                # Create bins for stratification
                y_binned = self._create_stratification_bins(y_values, self.n_bins)
                
                # Use stratified split
                strat_split = StratifiedShuffleSplit(
                    n_splits=1, 
                    test_size=self.test_size, 
                    random_state=self.random_state
                )
                
                for train_idx, test_idx in strat_split.split(X_values, y_binned):
                    self.X_train, self.X_test = X_values[train_idx], X_values[test_idx]
                    self.y_train, self.y_test = y_values[train_idx], y_values[test_idx]
                    
                print(f"📊 Stratification bins: {self.n_bins}")
                print(f"📈 Target distribution in train: {np.unique(y_binned[train_idx], return_counts=True)}")
                print(f"📊 Target distribution in test: {np.unique(y_binned[test_idx], return_counts=True)}")
                
            else:
                # Standard random split
                self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
                    X_values, y_values, test_size=self.test_size, random_state=self.random_state
                )
            
            # Scale features
            self.X_train_scaled = self.scaler.fit_transform(self.X_train)
            self.X_test_scaled = self.scaler.transform(self.X_test)
            
            # Apply data augmentation if enabled
            if self.augment_data:
                self.X_train_scaled, self.y_train = self.add_data_augmentation(
                    self.X_train_scaled, self.y_train, self.noise_factor
                )
            
        except Exception as e:
            print(f"Error in enhanced data preparation: {e}")
            # Fallback to standard preparation
            self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
                self.X, self.y, test_size=self.test_size, random_state=self.random_state
            )
            self.X_train_scaled = self.scaler.fit_transform(self.X_train)
            self.X_test_scaled = self.scaler.transform(self.X_test)

    def _prepare_data(self):
        """Prepare and scale the data for neural network"""
        try:
            # Convert to numpy arrays to avoid any pandas issues
            if hasattr(self.X, 'values'):
                X_values = self.X.values
            else:
                X_values = np.array(self.X)
                
            if hasattr(self.y, 'values'):
                y_values = self.y.values
            else:
                y_values = np.array(self.y)
            
            # Ensure y is 1D
            y_values = y_values.ravel()
            
            # Split the data
            self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
                X_values, y_values, test_size=self.test_size, random_state=self.random_state
            )
            
            # Scale features
            self.X_train_scaled = self.scaler.fit_transform(self.X_train)
            self.X_test_scaled = self.scaler.transform(self.X_test)
            
        except Exception as e:
            print(f"Error in data preparation: {e}")
            # Fallback
            self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
                self.X, self.y, test_size=self.test_size, random_state=self.random_state
            )
            self.X_train_scaled = self.scaler.fit_transform(self.X_train)
            self.X_test_scaled = self.scaler.transform(self.X_test)
    
    def create_model(self, architecture='standard', input_dim=None):
        """Create neural network model with different architectures"""
        if input_dim is None:
            input_dim = self.X_train_scaled.shape[1]
        
        model = Sequential()
        
        if architecture == 'simple':
            # Simple architecture
            model.add(Dense(64, activation='relu', input_shape=(input_dim,)))
            model.add(Dropout(0.2))
            model.add(Dense(32, activation='relu'))
            model.add(Dropout(0.2))
            model.add(Dense(1, activation='linear'))
            
        elif architecture == 'standard':
            # Standard architecture
            model.add(Dense(128, activation='relu', input_shape=(input_dim,),
                          kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)))
            model.add(BatchNormalization())
            model.add(Dropout(0.3))
            
            model.add(Dense(64, activation='relu', 
                          kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)))
            model.add(BatchNormalization())
            model.add(Dropout(0.3))
            
            model.add(Dense(32, activation='relu'))
            model.add(Dropout(0.2))
            
            model.add(Dense(1, activation='linear'))
            
        elif architecture == 'deep':
            # Deep architecture
            model.add(Dense(256, activation='relu', input_shape=(input_dim,),
                          kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)))
            model.add(BatchNormalization())
            model.add(Dropout(0.4))
            
            model.add(Dense(128, activation='relu',
                          kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)))
            model.add(BatchNormalization())
            model.add(Dropout(0.4))
            
            model.add(Dense(64, activation='relu'))
            model.add(Dropout(0.3))
            
            model.add(Dense(32, activation='relu'))
            model.add(Dropout(0.2))
            
            model.add(Dense(16, activation='relu'))
            model.add(Dropout(0.1))
            
            model.add(Dense(1, activation='linear'))
        
        elif architecture == 'wide':
            # Wide architecture
            model.add(Dense(512, activation='relu', input_shape=(input_dim,),
                          kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)))
            model.add(BatchNormalization())
            model.add(Dropout(0.5))
            
            model.add(Dense(256, activation='relu'))
            model.add(Dropout(0.3))
            
            model.add(Dense(128, activation='relu'))
            model.add(Dropout(0.2))
            
            model.add(Dense(1, activation='linear'))


        elif architecture == 'wide_regularized':
            # Wide architecture
            model.add(Dense(512, activation='relu', input_shape=(input_dim,),
                          kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)))
            model.add(BatchNormalization())
            model.add(Dropout(0.5))
            
            model.add(Dense(256, activation='relu'))
            model.add(Dropout(0.4))
            
            model.add(Dense(128, activation='relu'))
            model.add(Dropout(0.2))
            
            model.add(Dense(1, activation='linear'))
            
            # # Wide architecture
            # # Layer 1 - Much smaller with extreme regularization
            # model.add(Dense(128, activation='relu', input_shape=(input_dim,),
            #                kernel_regularizer=l1_l2(l1=1e-4, l2=5e-3)))  # 5x stronger L2
            # model.add(BatchNormalization())
            # model.add(Dropout(0.7))  # Extreme dropout
            
            # # Layer 2
            # model.add(Dense(64, activation='relu',
            #                kernel_regularizer=l1_l2(l1=1e-4, l2=2e-3)))
            # model.add(Dropout(0.6))
            
            # # Layer 3
            # model.add(Dense(32, activation='relu',
            #                kernel_regularizer=l1_l2(l1=1e-5, l2=1e-3)))
            # model.add(Dropout(0.5))
            
            # # Output
            # model.add(Dense(1, activation='linear'))
            
    
        
        return model
    
    def train_model(self, architecture='standard', epochs=500, batch_size=32, 
                   learning_rate=0.001, validation_split=0.2, verbose=0):
        """Train the neural network model"""
        print(f"Training Neural Network with {architecture} architecture...")
        print(f"Epochs: {epochs}, Batch size: {batch_size}, Learning rate: {learning_rate}")
        
        # Create model
        self.model = self.create_model(architecture=architecture)
        
        # Compile model
        optimizer = Adam(
            learning_rate=learning_rate,
            epsilon=1e-8,           # Crucial for small values
            clipnorm=1.0,           # Prevent gradient explosion

        )
        self.model.compile(
            optimizer=optimizer,
            loss='mae',
            metrics=['mae', 'mse']
        )
        
        # Callbacks
        # early_stopping = EarlyStopping(
        #     monitor='val_loss',
        #     patience=50,
        #     restore_best_weights=True,
        #     verbose=1
        # )
        
        # reduce_lr = ReduceLROnPlateau(
        #     monitor='val_loss',
        #     factor=0.5,
        #     patience=20,
        #     min_lr=1e-7,
        #     verbose=1
        # )
        
        early_stopping = EarlyStopping(
            monitor='val_loss',
            patience=30,  # Increased patience
            restore_best_weights=True,
            verbose=1,
            min_delta=0.0001  # Minimum change to qualify as improvement
        )
        
        # More aggressive learning rate reduction
        reduce_lr = ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=15,
            min_lr=1e-7,
            verbose=1
        )
        
        # Add model checkpointing
        checkpoint = tf.keras.callbacks.ModelCheckpoint(
            'best_model.h5',
            monitor='val_loss',
            save_best_only=True,
            verbose=1
        )
        #_______________________________________________________________
        
        # Train model
        self.history = self.model.fit(
            self.X_train_scaled, self.y_train,
            epochs=epochs,  # 500
            batch_size=batch_size, #42
            validation_split=validation_split, #0.2
            callbacks=[early_stopping, reduce_lr],
            verbose=verbose, #0
            shuffle=True
        )
        
        print("Neural Network training completed!")
        return self.history
    
    def evaluate_model(self):
        """Evaluate the trained model and return metrics"""
        if self.model is None:
            print("Model not trained yet. Please train the model first.")
            return None
        
        # Make predictions
        y_pred_train = self.model.predict(self.X_train_scaled).flatten()
        y_pred_test = self.model.predict(self.X_test_scaled).flatten()
        
        # Calculate metrics
        metrics = {
            'train_rmse': np.sqrt(mean_squared_error(self.y_train, y_pred_train)),
            'test_rmse': np.sqrt(mean_squared_error(self.y_test, y_pred_test)),
            'train_mse': mean_squared_error(self.y_train, y_pred_train),
            'test_mse': mean_squared_error(self.y_test, y_pred_test),
            'train_r2': r2_score(self.y_train, y_pred_train),
            'test_r2': r2_score(self.y_test, y_pred_test),
            'train_mae': mean_absolute_error(self.y_train, y_pred_train),
            'test_mae': mean_absolute_error(self.y_test, y_pred_test),
        }
        
        return metrics, y_pred_train, y_pred_test

# Enhanced training function
def run_enhanced_neural_network_analysis(X, y, architecture='standard', 
                                       compare_architectures=False, epochs=100, 
                                       learning_rate=0.001, batch_size=32,
                                       augment_data=False, noise_factor=0.01,
                                       stratify_data=False, n_bins=5):
    """
    Enhanced neural network analysis with data augmentation and stratification
    """
    print("🚀 Starting Enhanced Neural Network Analysis")
    print("=" * 60)
    print(f"📊 Data Augmentation: {augment_data}")
    print(f"🎯 Stratified Splitting: {stratify_data}")
    
    # Initialize enhanced analyzer
    nn_analyzer = EnhancedNeuralNetworkAnalyzer(
        X, y, 
        augment_data=augment_data,
        noise_factor=noise_factor,
        stratify_data=stratify_data,
        n_bins=n_bins
    )
    
    # Train the model
    history = nn_analyzer.train_model(
        architecture=architecture, 
        epochs=epochs, 
        learning_rate=learning_rate, 
        batch_size=batch_size
    )
    
    # Evaluate model
    metrics, _, _ = nn_analyzer.evaluate_model()
    
    print("\n" + "=" * 60)
    print("📊 ENHANCED NEURAL NETWORK RESULTS")
    print("=" * 60)
    print(f"Training R²:   {metrics['train_r2']:.4f}")
    print(f"Testing R²:    {metrics['test_r2']:.4f}")
    print(f"Testing RMSE:  {metrics['test_rmse']:.4f}")
    print(f"Testing MAE:   {metrics['test_mae']:.4f}")
    print(f"Data Augmentation: {augment_data}")
    print(f"Stratified Split: {stratify_data}")
    
    return history,  nn_analyzer, metrics, history

In [7]:
import xgboost as xgb
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import datetime
import json
warnings.filterwarnings('ignore')

# Set style for publication-ready plots
plt.style.use('seaborn-v0_8')
sns.set_palette("viridis")

class OptimizedModel:
    def __init__(self, parameters, X, y, cv_folds=5, random_state=42):
        self.X = X
        self.y = y.values.ravel() if hasattr(y, 'values') else y
        self.cv_folds = cv_folds
        self.random_state = random_state
        
        # Your optimized parameters
        self.best_params = parameters
        
        self.best_model = None
        self.oof_predictions = None
        self.oof_score = None
        self.model_info = {}
    
    def get_oof_predictions(self):
        """Generate Out-of-Fold predictions using optimized parameters"""
        kf = KFold(n_splits=self.cv_folds, shuffle=True, random_state=self.random_state)
        oof_preds = np.zeros(len(self.X))
        fold_scores = []
        
        for fold, (train_idx, val_idx) in enumerate(kf.split(self.X)):
            X_train, X_val = self.X.iloc[train_idx], self.X.iloc[val_idx]
            y_train, y_val = self.y[train_idx], self.y[val_idx]
            
            model = xgb.XGBRegressor(**self.best_params)
            model.fit(X_train, y_train, verbose=False)
            fold_preds = model.predict(X_val)
            oof_preds[val_idx] = fold_preds
            
            fold_r2 = r2_score(y_val, fold_preds)
            fold_scores.append(fold_r2)
        
        self.oof_predictions = oof_preds
        self.oof_score = r2_score(self.y, oof_preds)
        
        # Calculate additional metrics
        self.oof_mse = mean_squared_error(self.y, oof_preds)
        self.oof_rmse = np.sqrt(self.oof_mse)
        self.oof_mae = mean_absolute_error(self.y, oof_preds)
        self.oof_mape = np.mean(np.abs((self.y - oof_preds) / self.y)) * 100
        
        # Store fold scores
        self.fold_scores = fold_scores
        self.fold_scores_mean = np.mean(fold_scores)
        self.fold_scores_std = np.std(fold_scores)
        
        print(f"Out-of-Fold R²: {self.oof_score:.6f}")
        print(f"Out-of-Fold RMSE: {self.oof_rmse:.6f}")
        print(f"Out-of-Fold MAE: {self.oof_mae:.6f}")
        print(f"Out-of-Fold MAPE: {self.oof_mape:.6f}%")
        
        return oof_preds
    
    def train_final_model(self):
        """Train final model on all data"""
        self.best_model = xgb.XGBRegressor(**self.best_params)
        self.best_model.fit(self.X, self.y)
        
        # Calculate training metrics
        y_train_pred = self.best_model.predict(self.X)
        self.train_r2 = r2_score(self.y, y_train_pred)
        self.train_rmse = np.sqrt(mean_squared_error(self.y, y_train_pred))
        self.train_mae = mean_absolute_error(self.y, y_train_pred)
        
        return self.best_model
    
    def collect_model_info(self, X_test=None, y_test=None):
        """Collect all model information for saving"""
        self.model_info = {
            'timestamp': datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            'dataset_info': {
                'samples': self.X.shape[0],
                'features': self.X.shape[1],
                'feature_names': self.X.columns.tolist()
            },
            'hyperparameters': self.best_params,
            'cross_validation': {
                'folds': self.cv_folds,
                'fold_scores': self.fold_scores,
                'fold_mean_score': self.fold_scores_mean,
                'fold_std_score': self.fold_scores_std
            },
            'out_of_fold_metrics': {
                'r2_score': self.oof_score,
                'rmse': self.oof_rmse,
                'mse': self.oof_mse,
                'mae': self.oof_mae,
                'mape': self.oof_mape
            },
            'training_metrics': {
                'r2_score': self.train_r2,
                'rmse': self.train_rmse,
                'mae': self.train_mae
            },
            'feature_importance': dict(zip(self.X.columns, self.best_model.feature_importances_))
        }
        
        # Add test metrics if available
        if X_test is not None and y_test is not None:
            y_test_pred = self.best_model.predict(X_test)
            self.model_info['test_metrics'] = {
                'r2_score': r2_score(y_test, y_test_pred),
                'rmse': np.sqrt(mean_squared_error(y_test, y_test_pred)),
                'mae': mean_absolute_error(y_test, y_test_pred)
            }
        
        # Add plot data
        self.model_info['plot_data'] = {
            'actual_values': self.y.tolist(),
            'predicted_values': self.oof_predictions.tolist(),
            'residuals': (self.y - self.oof_predictions).tolist()
        }



def create_plots(model, y):
    """Create publication-ready compact plots"""
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
    
    # 1. Out-of-Fold Predictions vs Actual
    oof_preds = model.oof_predictions
    ax1.scatter(y, oof_preds, alpha=0.6, s=30)
    max_val = max(np.max(y), np.max(oof_preds))
    min_val = min(np.min(y), np.min(oof_preds))
    ax1.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2)
    ax1.set_xlabel('Actual Values')
    ax1.set_ylabel('Out-of-Fold Predictions')
    ax1.set_title(f'A) OOF Predictions vs Actual\nR² = {model.oof_score:.4f}', fontweight='bold')
    ax1.grid(True, alpha=0.3)
    
    # 2. Residuals Plot
    residuals = y - oof_preds
    ax2.scatter(oof_preds, residuals, alpha=0.6, s=30)
    ax2.axhline(y=0, color='r', linestyle='--', linewidth=2)
    ax2.set_xlabel('Predicted Values')
    ax2.set_ylabel('Residuals')
    ax2.set_title(f'B) Residuals Analysis\nRMSE = {model.oof_rmse:.4f}', fontweight='bold')
    ax2.grid(True, alpha=0.3)
    
    # 3. Feature Importance
    if model.best_model is not None:
        feature_importance = model.best_model.feature_importances_
        feature_names = model.X.columns
        
        importance_df = pd.DataFrame({
            'Feature': feature_names,
            'Importance': feature_importance
        }).sort_values('Importance', ascending=False)
        
        top_features = importance_df.head(15)
        ax3.barh(top_features['Feature'], top_features['Importance'])
        ax3.set_xlabel('Feature Importance Score')
        ax3.set_title('C) Top Feature Importance', fontweight='bold')
        ax3.grid(True, alpha=0.3)
    
    # 4. Prediction Distribution
    ax4.hist(y, bins=30, alpha=0.7, label='Actual', color='blue')
    ax4.hist(oof_preds, bins=30, alpha=0.7, label='Predicted', color='orange')
    ax4.set_xlabel('Values')
    ax4.set_ylabel('Frequency')
    ax4.set_title('D) Distribution: Actual vs Predicted', fontweight='bold')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'model_plots_{datetime.datetime.now().strftime("%Y%m%d_%H%M%S")}.png', dpi=300, bbox_inches='tight')
    plt.show()


# Usage in your main function:
def run_optimized_model_with_data_export(parameters, X, y, X_test=None, y_test=None):
    """Run complete analysis and export data in Excel-friendly format"""
    # Data preparation
    X_clean = X.apply(pd.to_numeric, errors='coerce')
    y_clean = y.values.ravel() if hasattr(y, 'values') else y
    
    print(f"Dataset: {X_clean.shape[0]} samples, {X_clean.shape[1]} features")
    
    # Create and run model
    model = OptimizedModel(parameters, X_clean, y_clean, cv_folds=5, random_state=42)
    oof_preds = model.get_oof_predictions()
    final_model = model.train_final_model()
    
    # Collect model information
    model.collect_model_info(X_test, y_test)
    
    # Create plots
    # create_plots(model, y_clean)
    
    # RESEARCH PAPER R² VALUE
    print(f"\n📊 RESEARCH PAPER R²: {model.oof_score:.6f}")
    
    return model, final_model
    

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np

def lgbModel(X, y):
    y_array = y.values.ravel()
    # ----------------------------------------------------
    # 1. Create approximately equal-size bins using qcut
    # ----------------------------------------------------
    n_bins = 15      #setA 15, 
    y_binned, bins_used = pd.qcut(y_array, q=n_bins, labels=False, retbins=True, duplicates='drop')
    
    # ----------------------------------------------------
    # 1. Train-test split
    # ----------------------------------------------------
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify = y_binned
    )
    
    print(f"Training set: {X_train.shape}")
    print(f"Testing set: {X_test.shape}")
    
    # ----------------------------------------------------
    # 2. Create LightGBM model
    # ----------------------------------------------------
    model = lgb.LGBMRegressor(
        n_estimators=15000,
        learning_rate=0.01,
        max_depth=5,         #setA 5, 
        num_leaves=32,  # Rule of thumb: 2^max_depth = 16 for depth=4
        random_state=42,
        n_jobs=-1,
        verbose=-1,  # Suppress output
        # reg_alpha=0,              # optional, set to 0 if not needed
    
        # reg_lambda=18,
        subsample_freq=1,          # frequency of bagging
        min_split_gain=0.0,        # randomness similar to CatBoost's random_strength
    )
    
    # ----------------------------------------------------
    # 3. Fit model with early stopping
    # ----------------------------------------------------
    model.fit(
        X_train, 
        y_train,
        eval_set=[(X_test, y_test)],
        eval_metric='l1',
        callbacks=[
            lgb.log_evaluation(50),  # Print every 50 iterations
            lgb.early_stopping(100)  # Stop if no improvement for 100 iterations
        ]
    )
    
    # # ----------------------------------------------------
    # # 4. Predictions
    # # ----------------------------------------------------
    # y_pred_train = model.predict(X_train)
    # y_pred_test = model.predict(X_test)
    
    # # ----------------------------------------------------
    # # 5. Metrics
    # # ----------------------------------------------------
    # r2_train = r2_score(y_train, y_pred_train)
    # r2_test = r2_score(y_test, y_pred_test)
    # mae = mean_absolute_error(y_test, y_pred_test)
    # rmse = np.sqrt(mean_squared_error(y_test, y_pred_test))
    
    # print("\n📌 Model Performance:")
    # print(f"Train R²: {r2_train:.4f}")
    # print(f"Test  R²: {r2_test:.4f}")
    # print(f"MAE:       {mae:.4f}")
    # print(f"RMSE:      {rmse:.4f}")
    
    # # ----------------------------------------------------
    # # 6. Simple learning curve plot
    # # ----------------------------------------------------
    # available_metrics = list(model.evals_result_['valid_0'].keys())
    # # Try to find MAE/L1 metric
    # metric_key = None
    # for key in ['l1', 'mae', 'mean_absolute_error']:
    #     if key in available_metrics:
    #         metric_key = key
    #         break
        
    #     if metric_key:
    #         val_scores = model.evals_result_['valid_0'][metric_key]
    # if 'valid_0' in model.evals_result_:
    #     plt.figure(figsize=(7,5))
    #     plt.plot(model.evals_result_['valid_0'][metric_key])
    #     plt.title("Validation RMSE vs Iterations")
    #     plt.xlabel("Iterations")
    #     plt.ylabel("MAE")
    #     plt.grid(True)
    #     plt.show()
    
    # # ----------------------------------------------------
    # # 7. Feature importance (optional simple plot)
    # # ----------------------------------------------------
    # plt.figure(figsize=(10, 6))
    # lgb.plot_importance(model, max_num_features=15, height=0.8)
    # plt.title("Feature Importance")
    # plt.tight_layout()
    # plt.show()

    return model


In [28]:
import numpy as np
import pandas as pd
import joblib
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.base import BaseEstimator, RegressorMixin
import warnings
warnings.filterwarnings('ignore')

# ---------- Import or define your base model classes ----------
# Assuming your OptimizedLightGBM, OptimizedModel, EnhancedNeuralNetworkAnalyzer 
# are already defined. If not, they are reproduced below.

# ============================================================================
# Placeholder for your existing model classes (they should be in your environment)
# If missing, uncomment the following sections or import them.
# ============================================================================
# (I'll include minimal working versions of your classes here for completeness)

import datetime
from scipy import stats
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------- LightGBM Class -----------------
class OptimizedLightGBM:
    def __init__(self, parameters, X, y, cv_folds=5, random_state=42, stratify_bins=15):
        self.X = X
        self.y = y.values.ravel() if hasattr(y, 'values') else y
        self.cv_folds = cv_folds
        self.random_state = random_state
        self.stratify_bins = stratify_bins
        self.best_params = parameters
        self.best_model = None
        self.oof_predictions = None
        self.oof_score = None
        self.model_info = {}
    
    def get_oof_predictions(self):
        kf = KFold(n_splits=self.cv_folds, shuffle=True, random_state=self.random_state)
        oof_preds = np.zeros(len(self.X))
        fold_scores = []
        stratify = None
        if self.stratify_bins is not None:
            stratify, _ = pd.qcut(self.y, q=self.stratify_bins, labels=False, retbins=True, duplicates='drop')
        for fold, (train_idx, val_idx) in enumerate(kf.split(self.X, stratify)):
            X_tr, X_val = self.X.iloc[train_idx], self.X.iloc[val_idx]
            y_tr, y_val = self.y[train_idx], self.y[val_idx]
            model = lgb.LGBMRegressor(**self.best_params)
            model.fit(X_tr, y_tr,
                      eval_set=[(X_val, y_val)],
                      eval_metric='l1',
                      callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
            oof_preds[val_idx] = model.predict(X_val)
            fold_scores.append(r2_score(y_val, oof_preds[val_idx]))
        self.oof_predictions = oof_preds
        self.oof_score = r2_score(self.y, oof_preds)
        self.oof_rmse = np.sqrt(mean_squared_error(self.y, oof_preds))
        self.oof_mae = mean_absolute_error(self.y, oof_preds)
        self.oof_mape = np.mean(np.abs((self.y - oof_preds) / (self.y + 1e-8))) * 100
        return oof_preds
    
    def train_final_model(self):
        self.best_model = lgb.LGBMRegressor(**self.best_params)
        self.best_model.fit(self.X, self.y)
        return self.best_model

# ---------------- XGBoost Class -----------------
class OptimizedModel:
    def __init__(self, parameters, X, y, cv_folds=5, random_state=42):
        self.X = X
        self.y = y.values.ravel() if hasattr(y, 'values') else y
        self.cv_folds = cv_folds
        self.random_state = random_state
        self.best_params = parameters
        self.best_model = None
        self.oof_predictions = None
        self.oof_score = None
    
    def get_oof_predictions(self):
        kf = KFold(n_splits=self.cv_folds, shuffle=True, random_state=self.random_state)
        oof_preds = np.zeros(len(self.X))
        for fold, (train_idx, val_idx) in enumerate(kf.split(self.X)):
            X_tr, X_val = self.X.iloc[train_idx], self.X.iloc[val_idx]
            y_tr, y_val = self.y[train_idx], self.y[val_idx]
            model = xgb.XGBRegressor(**self.best_params)
            model.fit(X_tr, y_tr, verbose=False)
            oof_preds[val_idx] = model.predict(X_val)
        self.oof_predictions = oof_preds
        self.oof_score = r2_score(self.y, oof_preds)
        self.oof_rmse = np.sqrt(mean_squared_error(self.y, oof_preds))
        self.oof_mae = mean_absolute_error(self.y, oof_preds)
        self.oof_mape = np.mean(np.abs((self.y - oof_preds) / (self.y + 1e-8))) * 100
        return oof_preds
    
    def train_final_model(self):
        self.best_model = xgb.XGBRegressor(**self.best_params)
        self.best_model.fit(self.X, self.y)
        return self.best_model

# ---------------- Neural Network Class -----------------
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.regularizers import l1_l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

class EnhancedNeuralNetworkAnalyzer:
    def __init__(self, X, y, test_size=0.2, random_state=42, augment_data=False,
                 noise_factor=0.01, stratify_data=False, n_bins=5):
        self.X = X
        self.y = y.values.ravel() if hasattr(y, 'values') else y
        self.test_size = test_size
        self.random_state = random_state
        self.augment_data = augment_data
        self.noise_factor = noise_factor
        self.stratify_data = stratify_data
        self.n_bins = n_bins
        self.scaler = StandardScaler()
        self.history = None
        self.model = None
        self._prepare_data()
    
    def _prepare_data(self):
        X_values = self.X.values if hasattr(self.X, 'values') else np.array(self.X)
        y_values = self.y
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            X_values, y_values, test_size=self.test_size, random_state=self.random_state
        )
        self.X_train_scaled = self.scaler.fit_transform(self.X_train)
        self.X_test_scaled = self.scaler.transform(self.X_test)
    
    def create_model(self, architecture='standard', input_dim=None):
        if input_dim is None:
            input_dim = self.X_train_scaled.shape[1]
        model = Sequential()
        if architecture == 'simple':
            model.add(Dense(64, activation='relu', input_dim=input_dim))
            model.add(Dropout(0.2))
            model.add(Dense(32, activation='relu'))
            model.add(Dropout(0.2))
            model.add(Dense(1))
        else:  # standard
            model.add(Dense(128, activation='relu', input_dim=input_dim,
                          kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)))
            model.add(BatchNormalization())
            model.add(Dropout(0.3))
            model.add(Dense(64, activation='relu', kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)))
            model.add(BatchNormalization())
            model.add(Dropout(0.3))
            model.add(Dense(32, activation='relu'))
            model.add(Dropout(0.2))
            model.add(Dense(1))
        return model
    
    def train_model(self, architecture='standard', epochs=100, batch_size=32,
                    learning_rate=0.001, validation_split=0.2, verbose=0):
        self.model = self.create_model(architecture, self.X_train_scaled.shape[1])
        self.model.compile(optimizer=Adam(learning_rate=learning_rate),
                           loss='mae', metrics=['mae'])
        early_stop = EarlyStopping(monitor='val_loss', patience=30, restore_best_weights=True)
        reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=15, min_lr=1e-7)
        self.history = self.model.fit(self.X_train_scaled, self.y_train,
                                      epochs=epochs, batch_size=batch_size,
                                      validation_split=validation_split,
                                      callbacks=[early_stop, reduce_lr],
                                      verbose=verbose)
        return self.history
    
    def evaluate_model(self):
        y_pred_train = self.model.predict(self.X_train_scaled).flatten()
        y_pred_test = self.model.predict(self.X_test_scaled).flatten()
        metrics = {
            'train_r2': r2_score(self.y_train, y_pred_train),
            'test_r2': r2_score(self.y_test, y_pred_test),
            'train_rmse': np.sqrt(mean_squared_error(self.y_train, y_pred_train)),
            'test_rmse': np.sqrt(mean_squared_error(self.y_test, y_pred_test)),
            'train_mae': mean_absolute_error(self.y_train, y_pred_train),
            'test_mae': mean_absolute_error(self.y_test, y_pred_test),
        }
        return metrics, y_pred_train, y_pred_test

# ============================================================================
# STACKING ENSEMBLE CLASS (supports multiple meta-models)
# ============================================================================
class StackingEnsemble:
    def __init__(self, lgb_params, xgb_params, nn_params, meta_model_type='xgboost',
                 meta_params=None, n_folds=5, random_state=42, stratify_bins=15):
        self.lgb_params = lgb_params
        self.xgb_params = xgb_params
        self.nn_params = nn_params
        self.meta_model_type = meta_model_type
        self.meta_params = meta_params if meta_params else {}
        self.n_folds = n_folds
        self.random_state = random_state
        self.stratify_bins = stratify_bins
        self.base_models = {}
        self.meta_model = None
        self.oof_predictions = {}
        self.test_predictions = {}
        self.metrics = {'train': {}, 'test': {}}
        self.scalers = {}   # store scalers for NN

    def _clean_data(self, X, y):
        X_clean = X.apply(pd.to_numeric, errors='coerce') if hasattr(X, 'apply') else X
        y_clean = y.values.ravel() if hasattr(y, 'values') else np.array(y).ravel()
        return X_clean, y_clean

    def _get_lightgbm_oof(self, X, y, X_test):
        kf = KFold(n_splits=self.n_folds, shuffle=True, random_state=self.random_state)
        oof_preds = np.zeros(len(X))
        test_preds = np.zeros(len(X_test) if X_test is not None else 0)
        if self.stratify_bins is not None:
            stratify, _ = pd.qcut(y, q=self.stratify_bins, labels=False, retbins=True, duplicates='drop')
        else:
            stratify = None
        for fold, (train_idx, val_idx) in enumerate(kf.split(X, stratify)):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y[train_idx], y[val_idx]
            model = lgb.LGBMRegressor(**self.lgb_params)
            model.fit(X_tr, y_tr,
                      eval_set=[(X_val, y_val)],
                      eval_metric='l1',
                      callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
            oof_preds[val_idx] = model.predict(X_val)
            if X_test is not None:
                test_preds += model.predict(X_test) / self.n_folds
        final_model = lgb.LGBMRegressor(**self.lgb_params)
        final_model.fit(X, y)
        self.base_models['LightGBM'] = final_model
        return oof_preds, test_preds

    def _get_xgboost_oof(self, X, y, X_test):
        kf = KFold(n_splits=self.n_folds, shuffle=True, random_state=self.random_state)
        oof_preds = np.zeros(len(X))
        test_preds = np.zeros(len(X_test) if X_test is not None else 0)
        for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y[train_idx], y[val_idx]
            model = xgb.XGBRegressor(**self.xgb_params)
            model.fit(X_tr, y_tr, verbose=False)
            oof_preds[val_idx] = model.predict(X_val)
            if X_test is not None:
                test_preds += model.predict(X_test) / self.n_folds
        final_model = xgb.XGBRegressor(**self.xgb_params)
        final_model.fit(X, y)
        self.base_models['XGBoost'] = final_model
        return oof_preds, test_preds

    def _get_neuralnet_oof(self, X, y, X_test):
        kf = KFold(n_splits=self.n_folds, shuffle=True, random_state=self.random_state)
        oof_preds = np.zeros(len(X))
        test_preds = np.zeros(len(X_test) if X_test is not None else 0)
        input_dim = X.shape[1]
        # We'll store scalers per fold for consistency (not strictly needed for OOF)
        for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
            X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
            y_tr, y_val = y[train_idx], y[val_idx]
            # Scale within fold
            scaler = StandardScaler()
            X_tr_scaled = scaler.fit_transform(X_tr)
            X_val_scaled = scaler.transform(X_val)
            if X_test is not None:
                X_test_scaled = scaler.transform(X_test)
            # Build model
            model = Sequential()
            arch = self.nn_params.get('architecture', 'standard')
            if arch == 'simple':
                model.add(Dense(64, activation='relu', input_dim=input_dim))
                model.add(Dropout(0.2))
                model.add(Dense(32, activation='relu'))
                model.add(Dropout(0.2))
                model.add(Dense(1))
            else:
                model.add(Dense(128, activation='relu', input_dim=input_dim,
                              kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)))
                model.add(BatchNormalization())
                model.add(Dropout(0.3))
                model.add(Dense(64, activation='relu', kernel_regularizer=l1_l2(l1=1e-5, l2=1e-4)))
                model.add(BatchNormalization())
                model.add(Dropout(0.3))
                model.add(Dense(32, activation='relu'))
                model.add(Dropout(0.2))
                model.add(Dense(1))
            model.compile(optimizer=Adam(learning_rate=self.nn_params.get('learning_rate', 0.001)),
                         loss='mae', metrics=['mae'])
            early_stop = EarlyStopping(monitor='val_loss', patience=30, restore_best_weights=True)
            reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=15, min_lr=1e-7)
            model.fit(X_tr_scaled, y_tr,
                      validation_data=(X_val_scaled, y_val),
                      epochs=self.nn_params.get('epochs', 100),
                      batch_size=self.nn_params.get('batch_size', 32),
                      callbacks=[early_stop, reduce_lr],
                      verbose=0)
            oof_preds[val_idx] = model.predict(X_val_scaled).flatten()
            if X_test is not None:
                test_preds += model.predict(X_test_scaled).flatten() / self.n_folds
        # Final model trained on all data with global scaler
        final_scaler = StandardScaler()
        X_scaled_all = final_scaler.fit_transform(X)
        final_model = Sequential()
        # same architecture as above (simple for final, could be improved)
        final_model.add(Dense(64, activation='relu', input_dim=input_dim))
        final_model.add(Dropout(0.2))
        final_model.add(Dense(32, activation='relu'))
        final_model.add(Dropout(0.2))
        final_model.add(Dense(1))
        final_model.compile(optimizer=Adam(learning_rate=self.nn_params.get('learning_rate', 0.001)),
                           loss='mae', metrics=['mae'])
        final_model.fit(X_scaled_all, y, epochs=self.nn_params.get('epochs', 100),
                        batch_size=self.nn_params.get('batch_size', 32), verbose=0)
        self.base_models['NeuralNet'] = final_model
        self.scalers['NeuralNet'] = final_scaler
        return oof_preds, test_preds

    def fit(self, X, y, X_test=None, y_test=None):
        X_clean, y_clean = self._clean_data(X, y)
        if X_test is not None:
            X_test_clean, y_test_clean = self._clean_data(X_test, y_test)
        else:
            X_test_clean = None
            y_test_clean = None

        # Generate OOF predictions
        lgb_oof, lgb_test = self._get_lightgbm_oof(X_clean, y_clean, X_test_clean)
        xgb_oof, xgb_test = self._get_xgboost_oof(X_clean, y_clean, X_test_clean)
        nn_oof, nn_test = self._get_neuralnet_oof(X_clean, y_clean, X_test_clean)

        self.oof_predictions = {
            'LightGBM': lgb_oof,
            'XGBoost': xgb_oof,
            'NeuralNet': nn_oof
        }
        self.test_predictions = {
            'LightGBM': lgb_test,
            'XGBoost': xgb_test,
            'NeuralNet': nn_test
        }

        # Meta-features
        meta_train = np.column_stack([lgb_oof, xgb_oof, nn_oof])

        # Train meta-model
        if self.meta_model_type == 'xgboost':
            self.meta_model = xgb.XGBRegressor(**self.meta_params)
        elif self.meta_model_type == 'ridge':
            self.meta_model = Ridge(**self.meta_params)
        elif self.meta_model_type == 'linear':
            self.meta_model = LinearRegression()
        else:
            raise ValueError(f"Unknown meta_model_type: {self.meta_model_type}")
        self.meta_model.fit(meta_train, y_clean)

        # Compute metrics on OOF (train)
        for name, preds in self.oof_predictions.items():
            self.metrics['train'][name] = self._compute_metrics(y_clean, preds)
        stacking_train = self.meta_model.predict(meta_train)
        self.metrics['train']['Stacking'] = self._compute_metrics(y_clean, stacking_train)
        self.oof_predictions['Stacking'] = stacking_train

        # Test set
        if X_test_clean is not None:
            meta_test = np.column_stack([self.test_predictions['LightGBM'],
                                         self.test_predictions['XGBoost'],
                                         self.test_predictions['NeuralNet']])
            stacking_test = self.meta_model.predict(meta_test)
            self.test_predictions['Stacking'] = stacking_test
            for name in ['LightGBM', 'XGBoost', 'NeuralNet', 'Stacking']:
                if name in self.test_predictions:
                    self.metrics['test'][name] = self._compute_metrics(y_test_clean, self.test_predictions[name])
        self._print_metrics()
        return self

    def _compute_metrics(self, y_true, y_pred):
        y_true = np.array(y_true).ravel()
        y_pred = np.array(y_pred).ravel()
        r2 = r2_score(y_true, y_pred)
        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mae = mean_absolute_error(y_true, y_pred)
        mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100
        return {'R2': r2, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape}

    def _print_metrics(self):
        print("\n" + "="*70)
        print("STACKING ENSEMBLE PERFORMANCE")
        print("="*70)
        if self.metrics['test']:
            print("\n--- TEST SET ---")
            df_test = pd.DataFrame.from_dict(self.metrics['test'], orient='index')
            print(df_test.round(6))
        print("\n--- TRAIN (OOF) SET ---")
        df_train = pd.DataFrame.from_dict(self.metrics['train'], orient='index')
        print(df_train.round(6))
        print("="*70 + "\n")

    def predict(self, X):
        X_clean, _ = self._clean_data(X, np.zeros(len(X)))
        # Base predictions
        lgb_pred = self.base_models['LightGBM'].predict(X_clean)
        xgb_pred = self.base_models['XGBoost'].predict(X_clean)
        # NN needs scaling
        nn_scaled = self.scalers['NeuralNet'].transform(X_clean)
        nn_pred = self.base_models['NeuralNet'].predict(nn_scaled).flatten()
        meta = np.column_stack([lgb_pred, xgb_pred, nn_pred])
        return self.meta_model.predict(meta)

    def save_model(self, filepath):
        """Save the entire stacking ensemble (base models, meta model, scalers) as a joblib."""
        import joblib
        ensemble_dict = {
            'base_models': self.base_models,
            'meta_model': self.meta_model,
            'scalers': self.scalers,
            'meta_model_type': self.meta_model_type,
            'lgb_params': self.lgb_params,
            'xgb_params': self.xgb_params,
            'nn_params': self.nn_params,
            'n_folds': self.n_folds,
            'random_state': self.random_state,
            'stratify_bins': self.stratify_bins
        }
        joblib.dump(ensemble_dict, filepath)
        print(f"✅ Stacking ensemble saved to {filepath}")

# ============================================================================
# TESTING FUNCTIONS FOR THE USER'S MAIN SCRIPT
# ============================================================================
def test_stacking_ensemble(ModelSet, meta_model_type='xgboost'):
    """
    Train a stacking ensemble on the dataset specified by ModelSet.
    Returns:
        stack_model: trained StackingEnsemble instance
        results: dict with test metrics if test set available
    """
    # Get data
    X, y_scaled = data_caller(ModelSet)   # y is already scaled by 100
    # Split into train/test
    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(X, y_scaled, test_size=0.2, random_state=42)

    # Hyperparameters (matching your previous models)
    lgb_params = {
        'n_estimators': 15000,
        'learning_rate': 0.01,
        'max_depth': 5,
        'num_leaves': 32,
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1,
        'subsample_freq': 1,
        'min_split_gain': 0.0,
        'reg_alpha': 0.0,
        'reg_lambda': 0.0,
        'boosting_type': 'gbdt',
        'objective': 'regression',
        'metric': 'l1'
    }
    xgb_params = {
        'n_estimators': 300,
        'learning_rate': 0.05,
        'max_depth': 6,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'random_state': 42,
        'verbosity': 0
    }
    nn_params = {
        'architecture': 'standard',
        'epochs': 100,
        'batch_size': 32,
        'learning_rate': 0.001
    }
    meta_params = {}
    if meta_model_type == 'xgboost':
        meta_params = {'n_estimators': 200, 'learning_rate': 0.01, 'max_depth': 3, 'random_state': 42}
    elif meta_model_type == 'ridge':
        meta_params = {'alpha': 1.0}

    stack = StackingEnsemble(lgb_params, xgb_params, nn_params,
                             meta_model_type=meta_model_type,
                             meta_params=meta_params,
                             n_folds=5, random_state=42, stratify_bins=15)
    stack.fit(X_train, y_train, X_test, y_test)
    # Save the best model (stacking ensemble)
    stack.save_model(f"stacking_ensemble_set_{ModelSet}_{meta_model_type}.joblib")
    results = stack.metrics['test'].get('Stacking', {})
    return stack, results

def compare_stacking_strategies():
    """Compare different meta-model types on ModelSet=3."""
    strategies = ['xgboost', 'ridge', 'linear']
    results = {}
    for strat in strategies:
        print(f"\n--- Testing meta-model: {strat.upper()} ---")
        _, res = test_stacking_ensemble(ModelSet, meta_model_type=strat)
        if 'R2' in res:
            results[strat] = res['R2']
        else:
            results[strat] = 0
    return results



# Calling Zone

In [20]:
# Data initiation 
import os
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
# =============================================================================================================================================================================================
def data_caller(ModelSet):
    #Data fixation for the model
    DoNormalize = True
    Z_ScoreScaler = True                       #If false then MinMax Scaler is used in the modelling
    ModelSet = ModelSet                        #1 denotes fixed base (SF all upto 4), 2 denotes SSI only case (SF upto 1 and inputheads include Layers) 3 denotes total datasets, (SF upto 1 , excluding soil layers ie Fixed similar)
    includeCategoricalData = True               #True sets the categorical datta intact and False triggers the one hot encoding to the categorical data
    X, y = dataPreparation(DoNormalize = DoNormalize,Z_ScoreScaler = Z_ScoreScaler, ModelSet = ModelSet ,includeCategoricalData = includeCategoricalData  )
    yscaled = y * 100
    
    X.shape
    # X.columns
    
    # # Execute display of data statistics
    # # =============================================================================================================================================================================================
    
    # # # Generate comprehensive statistics
    # # generate_complete_data_statistics(X, y)
    
    # # # Generate LaTeX table
    # # generate_latex_table(X, y)
    return X, yscaled 

In [31]:
import pickle
import joblib
# ============================================================================
# MAIN EXECUTION (as per your original snippet)
# ============================================================================

ModelSet = 1
if __name__ == "__main__":
    # Make sure data_caller is defined in your environment
    X, y_scaled = data_caller(ModelSet)
    
    print("\n" + "="*70)
    print("SEISMIC RESPONSE STACKING ENSEMBLE")
    print("="*70)
    
    # Option 1: Test single configuration (ModelSet=3, meta=ridge)
    print("\nOption 1: Testing single configuration")
    stack_model, results = test_stacking_ensemble(ModelSet, meta_model_type='ridge')
    
    # Option 2: Compare different stacking strategies
    print("\n\nOption 2: Comparing different stacking strategies")
    results_comparison = compare_stacking_strategies()
    
    # Compare with your previous results
    print(f"\n{'='*70}")
    print("COMPARISON WITH YOUR PREVIOUS RESULTS")
    print(f"{'='*70}")
    print(f"Your NN (Set A):           R² = 0.9346, RMSE = 0.1430")
    print(f"Your XGBoost (Set 3):      R² = 0.9363")
    best_r2 = max(results_comparison.values())
    best_meta = max(results_comparison, key=results_comparison.get)
    print(f"Best Stacking ({best_meta}): R² = {best_r2:.4f}")

    # The best model (stack_model) is already saved inside test_stacking_ensemble
    # If you want to explicitly save it again:
    stack_model.save_model(f"stacking_ensemble_set_{ModelSet}.joblib")


SEISMIC RESPONSE STACKING ENSEMBLE

Option 1: Testing single configuration
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[3165]	valid_0's l1: 0.0794075
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1972]	valid_0's l1: 0.0804804
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1264]	valid_0's l1: 0.100817
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[735]	valid_0's l1: 0.0768863
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[2981]	valid_0's l1: 0.0848358
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step 
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
6/6 ━━━━━━━━━━━━━━━